# LSTM–XGBoost Residual Correction

This notebook keeps the same preprocessing pipeline as the baseline study:
- chronological train/validation split
- Box-Cox transform on precipitation
- Min-Max scaling
- 70-day lag window for sequential forecasting

The difference is the modeling setup:
- Model 1: LSTM baseline
- Model 2: XGBoost 
- Model 3: LSTM–XGBoost residual correction, where XGBoost learns the residual target defined as:
  actual - LSTM_prediction

The residual target is built using chronological validation predictions from the LSTM, so the correction is learned from the same time-ordered validation regime.

In [ ]:
# Imports
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterSampler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

import xgboost as xgb
from numpy import array

In [ ]:
# Load precipitation dataset
csv_candidates = ['content/2619492.csv', '2619492.csv', 'Data/2619492.csv']
csv_path = next((p for p in csv_candidates if Path(p).exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find 2619492.csv in expected paths.')

rainfall = pd.read_csv(csv_path)
rainfall['DATE'] = pd.to_datetime(rainfall['DATE'], dayfirst=True)
rainfall = rainfall.set_index('DATE')
rainfall.head(3)

In [ ]:
# Histogram of precipitation values
plt.figure(figsize=(11, 6))
plt.rc('font', size=18)
plt.rc('axes', titlesize=20)
plt.rc('axes', labelsize=18)
plt.rc('xtick', labelsize=16)
plt.rc('ytick', labelsize=16)
plt.rc('legend', fontsize=18)
series = rainfall['PRCP']
series.hist(bins=50)
plt.title('Precipitation Distribution')
plt.xlabel('Precipitation')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Box-Cox and inverse Box-Cox helpers

def boxcox1p(data):
    p1data = data + 1.0
    bcdata, bclambda = boxcox(p1data)
    return bcdata, bclambda


def inv_boxcox1p(bcdata, bclambda):
    data = inv_boxcox(bcdata, bclambda) - 1.0
    return data

In [ ]:
# Box-Cox transformation on precipitation
inv_data, lam_value = boxcox1p(rainfall['PRCP'].astype(float))
plt.hist(inv_data, bins=100)
plt.title('Box-Cox transformed precipitation')
plt.show()

In [ ]:
# Min-Max scaling
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = inv_data.reshape(-1, 1)
scaler.fit(scaled_data)
rain_data = scaler.transform(scaled_data)[:, 0]
print(f'Rain data length: {len(rain_data)}')

In [ ]:
# Windowing function

def split_sequences(series, window):
    x, y = [], []
    for i in range(len(series)):
        end = i + window
        if end > len(series) - 1:
            break
        seq_x, seq_y = series[i:end], series[end]
        x.append(seq_x)
        y.append(seq_y)
    return array(x), array(y)

In [ ]:
# Chronological split: train + validation (same as baseline)
train_cutoff = 8988
window_size = 70

train_data = rain_data[:train_cutoff]
valid_data = rain_data[train_cutoff - window_size:10988]

x_train, y_train = split_sequences(train_data, window_size)
x_valid, y_valid = split_sequences(valid_data, window_size)

x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
x_valid = np.reshape(x_valid, (x_valid.shape[0], x_valid.shape[1], 1))

print(f'Train sequences: {x_train.shape[0]}')
print(f'Validation sequences: {x_valid.shape[0]}')

In [ ]:
# Model 1: LSTM baseline
model = Sequential()
model.add(LSTM(32, input_shape=(window_size, 1), return_sequences=True))
model.add(Dropout(0.5))
model.add(LSTM(32))
model.add(Dropout(0.5))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='relu'))

lr_schedule = tf.keras.callbacks.LearningRateScheduler(
    lambda epoch: 1e-7 * 10 ** (epoch / 40)
)
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-7, momentum=0.9)
model.compile(
    loss=tf.keras.losses.Huber(),
    optimizer=optimizer,
    metrics=['mean_squared_error'],
)

print(model.summary())

history = model.fit(
    x_train,
    y_train,
    epochs=100,
    batch_size=60,
    verbose=1,
    validation_data=(x_valid, y_valid),
    callbacks=[lr_schedule],
)

In [ ]:
# LSTM validation predictions on original scale
lstm_valid_pred_scaled = model.predict(x_valid, verbose=0).reshape(-1)

# Convert to original precipitation scale
lstm_valid_pred_orig = scaler.inverse_transform(lstm_valid_pred_scaled.reshape(-1, 1))[:, 0]
lstm_valid_pred_orig = inv_boxcox1p(lstm_valid_pred_orig, lam_value)

# Actual target on original scale
actual_valid_scaled = y_valid.reshape(-1, 1)
actual_valid_orig = scaler.inverse_transform(actual_valid_scaled)[:, 0]
actual_valid_orig = inv_boxcox1p(actual_valid_orig, lam_value)

rmse_lstm_valid = math.sqrt(mean_squared_error(actual_valid_orig, lstm_valid_pred_orig))
mae_lstm_valid = mean_absolute_error(actual_valid_orig, lstm_valid_pred_orig)
r2_lstm_valid = r2_score(actual_valid_orig, lstm_valid_pred_orig)

print(f'LSTM validation RMSE: {rmse_lstm_valid:.4f}')
print(f'LSTM validation MAE : {mae_lstm_valid:.4f}')
print(f'LSTM validation R2  : {r2_lstm_valid:.4f}')

In [ ]:
# Model 2: XGBoost baseline on same chronology

def build_xgb_feature_matrix(series, lag_window=70, rolling_windows=(7, 14), include_rolling=False):
    values = np.asarray(series, dtype=float)
    start_index = max(lag_window, max(rolling_windows, default=lag_window))
    x_rows = []
    y_rows = []
    target_indices = []

    for target_index in range(start_index, len(values)):
        lag_slice = values[target_index - lag_window:target_index]
        features = list(lag_slice)

        if include_rolling:
            for window in rolling_windows:
                trailing = values[target_index - window:target_index]
                features.append(float(np.mean(trailing)))
                features.append(float(np.std(trailing, ddof=0)))

        x_rows.append(features)
        y_rows.append(values[target_index])
        target_indices.append(target_index)

    x_array = np.asarray(x_rows, dtype=float)
    y_array = np.asarray(y_rows, dtype=float)
    target_indices = np.asarray(target_indices, dtype=int)

    lag_feature_names = [f'lag_{lag_window - i}' for i in range(lag_window)]
    if include_rolling:
        rolling_feature_names = []
        for window in rolling_windows:
            rolling_feature_names.extend([f'roll_mean_{window}', f'roll_std_{window}'])
        feature_names = lag_feature_names + rolling_feature_names
    else:
        feature_names = lag_feature_names

    return x_array, y_array, target_indices, feature_names


def summarize_feature_set(x_array, y_array, target_indices, train_cutoff=8988):
    train_mask = target_indices < train_cutoff
    valid_mask = (target_indices >= train_cutoff) & (target_indices < 10988)
    x_train = x_array[train_mask]
    y_train = y_array[train_mask]
    x_valid = x_array[valid_mask]
    y_valid = y_array[valid_mask]
    return x_train, y_train, x_valid, y_valid


x_lag_only, y_lag_only, idx_lag_only, lag_names = build_xgb_feature_matrix(
    rain_data[:10988],
    lag_window=70,
    rolling_windows=(7, 14),
    include_rolling=False,
)

x_train_xgb, y_train_xgb, x_valid_xgb, y_valid_xgb = summarize_feature_set(
    x_lag_only,
    y_lag_only,
    idx_lag_only,
    train_cutoff=train_cutoff,
)

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(x_train_xgb, y_train_xgb, verbose=False)

xgb_valid_pred_scaled = xgb_model.predict(x_valid_xgb)
xgb_valid_pred_orig = scaler.inverse_transform(xgb_valid_pred_scaled.reshape(-1, 1))[:, 0]
xgb_valid_pred_orig = inv_boxcox1p(xgb_valid_pred_orig, lam_value)

actual_valid_xgb_orig = scaler.inverse_transform(y_valid_xgb.reshape(-1, 1))[:, 0]
actual_valid_xgb_orig = inv_boxcox1p(actual_valid_xgb_orig, lam_value)

rmse_xgb_valid = math.sqrt(mean_squared_error(actual_valid_xgb_orig, xgb_valid_pred_orig))
mae_xgb_valid = mean_absolute_error(actual_valid_xgb_orig, xgb_valid_pred_orig)
r2_xgb_valid = r2_score(actual_valid_xgb_orig, xgb_valid_pred_orig)

print(f'XGBoost validation RMSE: {rmse_xgb_valid:.4f}')
print(f'XGBoost validation MAE : {mae_xgb_valid:.4f}')
print(f'XGBoost validation R2  : {r2_xgb_valid:.4f}')

## Model 3: LSTM–XGBoost residual correction

This is the new residual-training setup. The idea is:

- LSTM gives the primary temporal forecast
- XGBoost learns the residual: actual - LSTM_prediction
- The residual target is constructed from chronological validation predictions of the LSTM

This is a clean residual correction framework because the correction target is built from the same validation period used to evaluate the model.

In [ ]:
# Build residual target on the chronological validation window
# residual = actual - LSTM_prediction
# Both quantities are on the same scaled series and same time index.

# Ensure alignment between LSTM validation target and XGB validation target.
# LSTM uses x_valid from split_sequences(valid_data, window_size)
# XGBoost features are built from the same underlying series and valid index set.

# Convert LSTM validation predictions to the scaled series space
lstm_valid_pred_scaled = model.predict(x_valid, verbose=0).reshape(-1)

# Use actual target from XGBoost feature generation for the same chronological validation window
# Here actual_valid_xgb is the same actual target series aligned to the validation indices.
actual_valid_scaled = y_valid_xgb
residual_valid_scaled = actual_valid_scaled - lstm_valid_pred_scaled

# Train XGBoost on residuals using the same features that were used for the XGB baseline
residual_xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

residual_xgb.fit(x_valid_xgb, residual_valid_scaled, verbose=False)

# Predict residual correction on the same validation window
residual_valid_pred = residual_xgb.predict(x_valid_xgb)

# Final corrected forecast = LSTM prediction + residual correction
corrected_valid_scaled = lstm_valid_pred_scaled + residual_valid_pred

# Convert corrected output back to original precipitation scale
corrected_valid_orig = scaler.inverse_transform(corrected_valid_scaled.reshape(-1, 1))[:, 0]
corrected_valid_orig = inv_boxcox1p(corrected_valid_orig, lam_value)

actual_valid_final_orig = scaler.inverse_transform(actual_valid_scaled.reshape(-1, 1))[:, 0]
actual_valid_final_orig = inv_boxcox1p(actual_valid_final_orig, lam_value)

rmse_residual_valid = math.sqrt(mean_squared_error(actual_valid_final_orig, corrected_valid_orig))
mae_residual_valid = mean_absolute_error(actual_valid_final_orig, corrected_valid_orig)
r2_residual_valid = r2_score(actual_valid_final_orig, corrected_valid_orig)

print(f'LSTM + XGBoost residual validation RMSE: {rmse_residual_valid:.4f}')
print(f'LSTM + XGBoost residual validation MAE : {mae_residual_valid:.4f}')
print(f'LSTM + XGBoost residual validation R2  : {r2_residual_valid:.4f}')

In [ ]:
# Summary table for comparison
comparison = pd.DataFrame({
    'Model': ['LSTM baseline', 'XGBoost baseline', 'LSTM + XGBoost residual'],
    'RMSE': [rmse_lstm_valid, rmse_xgb_valid, rmse_residual_valid],
    'MAE': [mae_lstm_valid, mae_xgb_valid, mae_residual_valid],
    'R2': [r2_lstm_valid, r2_xgb_valid, r2_residual_valid],
})

print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

### Notes on interpretation

- The residual model is trained to predict the difference between the true target and the LSTM prediction.
- This is different from averaging two raw forecasts.
- It is a stronger residual-correction design because the XGBoost model explicitly learns the systematic errors left behind by the LSTM.
- If the residual signal is meaningful, the residual-corrected model should outperform the pure LSTM baseline.